# 🌙 09. LightGlue Deep Transformer Correspondence Matching

**Mission Context**: Adaptive transformer-based feature matching across extreme lunar illumination and viewpoint shifts.  
**Objectives**:
- Match uniformly distributed SuperPoint descriptors using LightGlue self- and cross-attention.
- Calculate Match Confidences and Match Density.
- Export `matches.csv` and `matches.json`.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import json

sys.path.append(str(Path.cwd().parent))
from lunar_core.config import load_config
from lunar_core.matching import LightGlueMatcher
from lunar_core.synthetic_data import LunarSyntheticGenerator

config = load_config()
gen = LunarSyntheticGenerator(size=(512, 512), seed=42)
pair = gen.generate_registered_pair(rotation_deg=12.0, scale=1.05, tx=30.0, ty=-20.0)

ref_img, src_img = pair["reference_image"], pair["source_image"]
raw_ref = np.load("outputs/features/features_reference.npz")
raw_src = np.load("outputs/features/features_source.npz")

matcher = LightGlueMatcher(filter_threshold=0.15)
match_res = matcher.match(
    {"keypoints": raw_src["keypoints"], "descriptors": raw_src["descriptors"]},
    {"keypoints": raw_ref["keypoints"], "descriptors": raw_ref["descriptors"]}
)

print(f"Total Matches Found: {match_res['total_matches']}")
print(f"Average Match Confidence: {match_res['average_confidence']}")
print(f"Match Density Ratio: {match_res['match_density']}")


In [ ]:
# Visualize LightGlue Matches
h, w = ref_img.shape
vis_match = np.zeros((h, w * 2, 3), dtype=np.uint8)
vis_match[:, :w] = cv2.cvtColor(src_img, cv2.COLOR_GRAY2BGR)
vis_match[:, w:] = cv2.cvtColor(ref_img, cv2.COLOR_GRAY2BGR)

for p_src, p_ref, conf in zip(match_res["matched_kpts0"][:100], match_res["matched_kpts1"][:100], match_res["confidences"][:100]):
    pt1 = (int(p_src[0]), int(p_src[1]))
    pt2 = (int(p_ref[0] + w), int(p_ref[1]))
    color = (0, int(255 * conf), int(255 * (1.0 - conf)))
    cv2.line(vis_match, pt1, pt2, color, 1, cv2.LINE_AA)

plt.figure(figsize=(16, 8))
plt.imshow(cv2.cvtColor(vis_match, cv2.COLOR_BGR2RGB))
plt.title(f"LightGlue Deep Matches (N={match_res['total_matches']}, Avg Conf: {match_res['average_confidence']})", fontsize=14, fontweight='bold')
plt.axis('off')

os.makedirs("outputs/visualizations", exist_ok=True)
plt.savefig("outputs/visualizations/09_lightglue_matches.png", dpi=300)
plt.show()


In [ ]:
# Export Matches
records = []
for i in range(len(match_res["matched_kpts0"])):
    records.append({
        "match_id": i,
        "src_x": round(float(match_res["matched_kpts0"][i, 0]), 3),
        "src_y": round(float(match_res["matched_kpts0"][i, 1]), 3),
        "ref_x": round(float(match_res["matched_kpts1"][i, 0]), 3),
        "ref_y": round(float(match_res["matched_kpts1"][i, 1]), 3),
        "confidence": round(float(match_res["confidences"][i]), 4)
    })

os.makedirs("outputs/matches", exist_ok=True)
df_matches = pd.DataFrame(records)
df_matches.to_csv("outputs/matches/matches.csv", index=False)

summary = {
    "total_matches": match_res["total_matches"],
    "average_confidence": match_res["average_confidence"],
    "match_density": match_res["match_density"]
}
with open("outputs/matches/matches.json", "w") as f:
    json.dump(summary, f, indent=4)

print("Exported outputs/matches/matches.csv and matches.json")
